In [5]:
import torch
from transformers import AutoModel, AutoTokenizer

In [6]:
MODEL_NAME = "vinai/phobert-base"

CACHE_DIR = "../../data/models/vinai-phobert"

phoBert = AutoModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)

# để model chuyển sang evaluation mode (do chỉ inference)
phoBert.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 45317.98it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

In [7]:
# INPUT TEXT MUST BE ALREADY WORD-SEGMENTED!
# sentence = 'Chúng_tôi là những nghiên_cứu_viên .'
sentence = 'Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .'

In [8]:
# Tokenization
encoded = tokenizer(
    sentence,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

In [9]:
input_ids = encoded["input_ids"][0]
attention_mask = encoded["attention_mask"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print(f"{'TOKEN':<25} {'ID':<10} {'MASK':<5}")
print("-" * 45)

for token, token_id, mask in zip(
    tokens,
    input_ids.tolist(),
    attention_mask.tolist()
):
    print(f"{token:<25} {token_id:<10} {mask:<5}")

TOKEN                     ID         MASK 
---------------------------------------------
<s>                       0          1    
Đặt                       8209       1    
vé                        826        1    
từ                        39         1    
TP                        253        1    
HCM                       383        1    
đi                        57         1    
Singapore                 1462       1    
để                        24         1    
công_tác                  247        1    
,                         4          1    
chị                       213        1    
Hoàng_@@                  1826       1    
Loan                      4138       1    
,                         4          1    
ở                         25         1    
phường                    557        1    
Xuân_Hoà                  25553      1    
,                         4          1    
bất_ngờ                   593        1    
vì                        90         1    
mức     

In [10]:
# Forward qua PhoBERT
with torch.no_grad():
    output = phoBert(**encoded)

token_embeddings = output.last_hidden_state

print("Token embeddings shape:")
print(token_embeddings.shape)

Token embeddings shape:
torch.Size([1, 70, 768])


[1, 73, 768]
 │   │    │
 │   │    └── mỗi token = vector 768 chiều
 │   └─────── 73 tokens
 └─────────── 1 sentence